# Thai Doc Classifier — run the 7B on a free GPU (Colab / Kaggle)

Runs `bench_local.py` with **Qwen2.5-VL-7B** on a free T4 (16 GB) GPU — the model your 6 GB laptop can't host.

**Before you start:** Runtime → Change runtime type → **GPU (T4)**.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available(), '| torch', torch.__version__)

## 1. Get the code
**Upload a zip** of the project folder (so `thaidoc_llm/`, `thaidoc/`, `bench_local.py`, and your `test-files/` are inside), then run the cell and pick the zip.

*Option B: clone from git* — uncomment the last lines and set your repo URL (private repos need a token).

In [ ]:
import os, zipfile, glob
from google.colab import files

up = files.upload()  # choose your project .zip
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/project')

# cd into the folder that actually contains thaidoc_llm/
root = next(os.path.dirname(p) for p in glob.glob('/content/project/**/thaidoc_llm', recursive=True))
os.chdir(root)
print('Working dir:', os.getcwd())
print('Has bench_local.py:', os.path.exists('bench_local.py'))

# --- Option B: git clone instead of uploading ---
# !git clone https://<TOKEN>@github.com/<you>/<repo>.git /content/project
# os.chdir('/content/project')

## 2. Install dependencies
Colab already has CUDA PyTorch — we only add the model libraries (this is why it works here but not on your CPU-only laptop torch).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils \
    openpyxl pillow opencv-python-headless scikit-learn matplotlib
print('deps installed')

## 3. Point at your test images
Uses the `test-files/` folder from your uploaded zip — the normal bench input. Make sure `test-files/` is included in the zip.

In [ ]:
import os

DATA_DIR = 'test-files'   # the real test images you included in the upload
exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp')
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(exts)]
print('images:', len(imgs), 'in', DATA_DIR)
for f in sorted(imgs):
    print(' -', f)

## 4. Run the benchmark on the 7B
First run downloads the 7B weights (~16 GB) into the Colab session — a few minutes on Colab's fast link. On a 16 GB T4 the 7B loads fully on-GPU in 4-bit (no CPU offload), ~2–4 s/image.

In [ ]:
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-7B-Instruct --dir $DATA_DIR

### Compare against the 3B (same data)

In [ ]:
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-3B-Instruct --dir $DATA_DIR

## 5. View / download the text report

In [ ]:
print(open('bench_report_transformers.txt', encoding='utf-8').read())
from google.colab import files
files.download('bench_report_transformers.txt')